# `Pattern 1` retrieval & generation - AI4RAG Notebook <font size="3">ver. 0.0.3</font>

Consider these tips for working with an auto-generated notebook:
- Notebook code generated with ai4rag will execute successfully. If you modify the notebook, we cannot guarantee it will run successfully.
- This RAG pattern is optimized for the original data set. The pattern might fail or produce sub-optimal results if used with different data. If you want to use a different data set, consider rerunning the ai4rag experiment to generate a new pattern. For more information, see {platform_link}.


<a id="content"></a>
## Notebook content

This notebook contains a python code for building retrieval & generation pattern. This notebook introduces commands for retrieving chunks, building prompt and generating answers.

Some familiarity with Python is helpful. This notebook uses Python 3.13.

## Notebook goals

- Test generated RAG pattern
- Evaluate generated RAG pattern

#### About Retrieval Augmented Generation
Retrieval Augmented Generation (RAG) is a versatile pattern that can unlock a number of use cases requiring factual recall of information, such as querying a knowledge base in natural language.

In its simplest form, RAG requires 3 steps:

- Index knowledge base passages (once)
- Retrieve relevant passage(s) from knowledge base (for every user query)
- Generate a response by feeding retrieved passage into a large language model (for every user query)

This notebook covers steps 2 & 3.

## Contents

This notebook contains the following parts:

**[Setup](#setup)**
<br>&nbsp;&nbsp;[Package installation](#package-installation)
<br>&nbsp;&nbsp;[Client initialization](#client-initialization)

**[RAG Pattern initialization](#rag-pattern-initialization)**
<br>&nbsp;&nbsp;[Embedding model initialization](#embedding-model)
<br>&nbsp;&nbsp;[Vector store initialization](#vector-store)
<br>&nbsp;&nbsp;[Retriever initialization](#retriever)
<br>&nbsp;&nbsp;[Foundation model initialization](#foundation-model)
<br>&nbsp;&nbsp;[RAG Pattern initialization](#rag-pattern)

**[Test RAG Pattern](#test-rag-pattern)**
<br>&nbsp;&nbsp;[Prepare benchmark data](#prepare-benchmark-data)
<br>&nbsp;&nbsp;[Generate response](#generate-response)
<br>&nbsp;&nbsp;[Evaluate RAG Pattern's response](#evaluate-rag-patterns-response)<br>


# Setup

## Package installation
Install ai4rag package with all its dependencies.

In [ ]:
!pip install "git+https://github.com/IBM/ai4rag.git@dev"

## Client initialization

Instantiate LlamaStackClient for communication with llama-stack.

In [ ]:
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url="http://localhost:8321")

# RAG Pattern initialization

## Embedding model

Embedding model is responsible for creating vectorized form of the given chunks.
It is required to use the same embedding model that was used for adding the documents / chunks to the vector store.
If different embedding model is used, results might not be optimal.

In [ ]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel

embedding_model = LSEmbeddingModel(
    client=client,
    model_id="ollama/nomic-embed-text:latest",
    params={
        "context_length": 8192,
        "embedding_dimension": 768,
    }
)

## Vector store

Instance of the vector store allows to communicate with given vector database using llama-stack.
`reuse_collection_name` parameter allows to use already existing collection.

In [ ]:
# Note! If you are just testing notebook, you can create index using this cell.

# from langchain_core.documents import Document
#
# from ai4rag.rag.vector_store.llama_stack import LSVectorStore
#
# vector_store = LSVectorStore(
#     client=client,
#     embedding_model=embedding_model,
#     provider_id="milvus",
# )
#
# vector_store.add_documents(
#     documents=[
#         Document(page_content="Being good to other people seems to be the meaning of life.", metadata={"document_id": "doc.txt"})
#     ]
# )
# vs_id = vector_store._ls_vs.id

vs_id = "SET IT MANUALLY"

In [ ]:
from ai4rag.rag.vector_store.llama_stack import LSVectorStore

vector_store = LSVectorStore(
    client=client,
    embedding_model=embedding_model,
    provider_id="milvus",
    reuse_collection_name=vs_id,
)

## Retriever

Retriever is responsible for retrieving relevant chunks from the vector store.

In [ ]:
from ai4rag.rag.retrieval.retriever import Retriever

retriever = Retriever(
    vector_store=vector_store,
    method="simple",
    number_of_chunks=5,
)

## Foundation model

Foundation model instance allows to communicate with the given LLM.
To configure foundation model with different parameters / prompt templates see the documentation.

In [ ]:
from ai4rag.search_space.src.model_props import get_system_message_text
from ai4rag.search_space.src.model_props import get_user_message_text
from ai4rag.search_space.src.model_props import get_context_template_text
from ai4rag.rag.foundation_models.foundation_model import LSFoundationModel

model_id = "ollama/llama3.2:3b"

foundation_model = LSFoundationModel(
    client=client,
    model_id=model_id,
    system_message_text=get_system_message_text(model_name=model_id),
    user_message_text=get_user_message_text(model_name=model_id),
    context_template_text=get_context_template_text(model_name=model_id)
)

## RAG Pattern

Compose instance of the RAG Pattern that can utilise all components to perform e2e retrieval augmented generation.

In [ ]:
from ai4rag.rag.template.rag_template import LlamaStackRAG

rag_pattern = LlamaStackRAG(
    foundation_model=foundation_model,
    retriever=retriever,
)

# Test RAG Pattern

## Prepare benchmark data

To properly evaluate RAG pattern's at least 1 record from benchmark data is required.

In [ ]:
import pandas as pd
from ai4rag.core.experiment.benchmark_data import BenchmarkData

# Here download or injection will be prepared.
# For testing purposes let's mock this behaviour and simply create one benchmark_data records.

raw_benchmark_data = [
    {
        "question": "What is the meaning of life?",
        "correct_answers": [
            "Being good to other people."
        ],
        "correct_answer_document_ids": ["doc.txt"]
    }
]

benchmark_data = BenchmarkData(pd.DataFrame(raw_benchmark_data))

## Generate response

Use prepared RAG pattern for response generation using retrieval augmented generation.

In [ ]:
response = rag_pattern.generate(
    question=benchmark_data.questions[0],
)

## Evaluate RAG Pattern's response

In [ ]:
from ai4rag.evaluator.base_evaluator import EvaluationData
from ai4rag.evaluator.unitxt_evaluator import UnitxtEvaluator, MetricType

evaluator = UnitxtEvaluator()

contexts = []
context_ids = []
for el in response["reference_documents"]:
    contexts.append(getattr(el, "page_content", None))
    context_ids.append(getattr(el, "metadata", {}).get("document_id"))

eval_data = EvaluationData(
    question=benchmark_data.questions[0],
    answer=response["answer"],
    contexts=contexts,
    context_ids=context_ids,
    ground_truths=benchmark_data.answers[0],
    question_id=benchmark_data.questions_ids[0],
    ground_truths_context_ids=benchmark_data.document_ids[0],
)

result = evaluator.evaluate_metrics(
    evaluation_data=[eval_data],
    metrics=[MetricType.ANSWER_CORRECTNESS, MetricType.FAITHFULNESS, MetricType.CONTEXT_CORRECTNESS]
)

result